<br>
<font size=6 color=#009999> 1 - Load Dataset </font> <br>
<br>

In [ ]:
!pip install natsort
!pip install tensorflow

In [ ]:
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
from natsort import natsorted 

print(os.getcwd())

domain1_path =  "data/Domain1_csv"
files = natsorted(os.listdir(domain1_path)) 
data = {
    'subjects': [],      # Index du sujet (1-10)
    'gesture_type': [],  # Type de geste (0-9)
    'repetition': [],    # Numéro de répétition (1-10)
    'trajectory': []     # Tableau numpy des positions [129, 3]
}


scaler = StandardScaler()

for filename in files:
    filepath = os.path.join(domain1_path, filename)
    df = pd.read_csv(filepath, header=None, names=["x", "y", "z", "t"],skiprows=1)

    # Extraire subject, gesture, repeat depuis le nom du fichier
    # Ex: Subject1-0-2.csv → subject=1, gesture=0, repeat=2
    parts = filename.replace(".csv", "").replace("Subject", "").split("-")
    subject_id = int(parts[0])
    gesture_id = int(parts[1])
    repetition_id = int(parts[2])

    data['subjects'].append(subject_id)
    data['gesture_type'].append(gesture_id)
    data['repetition'].append(repetition_id)
    # Add points, Standardize each data
    data['trajectory'].append(scaler.fit_transform(df.iloc[:, :3].values.astype(float)))

dataframe = pd.DataFrame(data)


In [ ]:
# import numpy as np
from sklearn import preprocessing
import numpy as np
# Import the dataset and discard non-quantitative features
#dataset01 = pd.read_csv("GestureRecognitionProject\Datasets_CSV\Domain1_csv\Subject1-0-1.csv") # todo other dataset

# Remove the time step (last column)
#dataset01 = dataset01.drop(["<t>"], axis=1)

# Observe the dataset
dataframe.info()
display(dataframe.describe())

#mean length for resizing for Deep method
lengths = [len(traj) for traj in data['trajectory']]
target_len = int(np.median(lengths))
print(target_len)

<br>
<font size=6 color=#009999> 2 - PCA (Principal Component Analysis) </font> <br>
<br>

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt

#### PCA #####
# Stack all points from all files
concatenated = np.vstack(data['trajectory'])  # shape: (total_points, 3)

# Step 2: Standardize all data (already done)
scaled_points = concatenated

# Step 3: Fit PCA on scaled data
pca = PCA(n_components=2)
pca.fit(scaled_points)

# Step 4: Apply PCA to each sample
#projected_samples = [pca.transform(seq) for seq in data['trajectory']]
data['trajectory'] = [pca.transform(seq) for seq in data['trajectory']]
# comment to choose if we want to go with PCA or only scale
#data['trajectory'] = projected_sample # PCA
#data['trajectory'] = [seq[:, :-1] for seq in data['trajectory']] # only scale (we take x,y)
#### PCA #####

labels = data['gesture_type']

#comment to change what is visualize in the graph
X_pca = np.vstack(scaled_points)

#X_pca = np.vstack(projected_samples) #bizarre mais global PCA rends moins bien visible les chiffres
fig, axs = plt.subplots(2, 5, figsize=(15, 6))
axs = axs.ravel()

for gesture in range(10):
    indices = [i for i, g in enumerate(data['gesture_type']) if g == gesture]
    for idx in indices[:9]:  # jusqu’à 5 répétitions par geste
        traj = data['trajectory'][idx]
        axs[gesture].plot(traj[:, 0], traj[:, 1], alpha=0.6)
    axs[gesture].set_title(f"Geste {gesture}")
    axs[gesture].axis('equal')

plt.tight_layout()
plt.show()

# Optional: build labels for each point (flattened version)
# Create label per point, repeating digit label for each point in that gesture
point_labels = np.concatenate([
    np.full(len(seq), label) for seq, label in zip(data['trajectory'], labels)
])

# Plot PCA points
fig = px.scatter(
    x=X_pca[:, 0], y=X_pca[:, 1],
    color=point_labels.astype(str),  # color by digit
    title="PCA of All Gesture Points",
    labels={'x': 'PC1', 'y': 'PC2'}
)
fig.show()

# Explained variance ratio
EVR = pca.explained_variance_ratio_
cumulEVR = np.zeros(3)
cumulEVR[1:] = np.cumsum(EVR)

# Cumulative variance plot
plt.plot(np.arange(3), cumulEVR)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance ratio")
plt.title("PCA Explained Variance")
plt.grid(True)
plt.show()


<br>
<font size=6 color=#009999> 3 - cross-validation </font> <br>
<br>

In [ ]:
from scipy.interpolate import interp1d

# Need fixed size of coordinate for Deep learning method
def resample_trajectory(traj, target_len=85):
    """Resample a (L, 2) trajectory to fixed length (target_len, 2)."""
    t_original = np.linspace(0, 1, len(traj))
    t_resampled = np.linspace(0, 1, target_len)
    f = interp1d(t_original, traj, axis=0)
    return f(t_resampled)


def Cross_validation(data, model, Deep=False, user_dependent=False,k=int):
    subjects = np.array(data['subjects'])
    gesture_types = np.array(data['gesture_type'])
    repetitions = np.array(data['repetition'])
    #Nee to have same size for np.array() -> either resample by interpolating or padd end util max lenght
    trajectories = np.array([resample_trajectory(traj) for traj in data['trajectory']])# shape: (N, 85, 3) with 85 the median of trajectory size
    all_y_test = []
    all_y_pred = []
    accuracies = []
    train_idx = 0
    test_idx = 0

    for leave_out in range(1,11):
        # Create train/test split
        if user_dependent:
            train_idx = repetitions != leave_out # leave one repetitions (1 to 10) (assumption we drop the same repetition for ALL user)
            test_idx = repetitions == leave_out
        else:
            train_idx = subjects != leave_out # leave subjects (1 to 10)
            test_idx = subjects == leave_out

        X_train = np.array([traj.flatten() for traj in trajectories[train_idx]])
        y_train = gesture_types[train_idx]

        X_test = np.array([traj.flatten() for traj in trajectories[test_idx]])
        y_test = gesture_types[test_idx]

        #print(np.shape(X_train))
        #print(np.shape(y_train))
        # Train and evaluate
        acc = 0
        y_pred = []
        if Deep:
            model.fit(X_train, to_categorical(y_train), epochs=20, batch_size=32, verbose=0)
            _, acc = model.evaluate(X_test, to_categorical(y_test), verbose=0)
            y_pred = np.argmax(model.predict(X_test), axis=1)
        else:
            
            for test_sample in X_test:
                pred = K_NN(X_train, y_train, test_sample, k)
                y_pred.append(pred)
            acc = accuracy_score(y_test, y_pred)
        all_y_test.extend(y_test)
        all_y_pred.extend(y_pred)
          
        accuracies.append(acc)



        if user_dependent:
            print(f"Fold (leave repetitions {leave_out} out): Accuracy = {acc:.4f}")
        else:
            print(f"Fold (leave subject {leave_out} out): Accuracy = {acc:.4f}")

    print(f"\nMean accuracy over 10 folds: {np.mean(accuracies):.4f}")
    print(f"Écart-type de l'accuracy : {np.std(accuracies):.4f}")
    return accuracies, all_y_test, all_y_pred
    


<br>
<font size=6 color=#009999> 4 - dynamic time warping et K_NN </font> <br>
<br>

In [ ]:
from collections import Counter
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from numba import jit, prange
from joblib import Parallel, delayed
import seaborn as sns

@jit(nopython=True, parallel=True)
#dynamic time warping
def dtw(s1,s2,window):
    s1= np.asarray(s1).reshape(-1, 2)
    s2= np.asarray(s2).reshape(-1, 2)
    n=len(s1)
    m=len(s2)
    w=max(window,abs(n - m))
    dtw_mat=np.full((n+1,m+1,),np.inf)
    dtw_mat[0, 0] = 0
    
    for i in range(1, n+1):
        for j in range(max(1, i - w), min(m + 1, i + w + 1)):
            cost = 0.0
            for k in range(s1.shape[1]):
                diff = s1[i-1, k] - s2[j-1, k]
                cost += diff * diff
            cost = np.sqrt(cost)
            dtw_mat[i, j] = cost + min(
                dtw_mat[i-1, j],
                dtw_mat[i, j-1],
                dtw_mat[i-1, j-1]
            )

    return dtw_mat[-1,-1]
    

#K_NN
def K_NN(train_data, train_labels, test_sample, k, n_jobs=-1):
    def compute_distance(i):
        return dtw(train_data[i], test_sample,5), train_labels[i]
    
    distances = Parallel(n_jobs=n_jobs)(
        delayed(compute_distance)(i) for i in range(len(train_data))
    )
    
    distances.sort(key=lambda x: x[0])
    k_nearest = [label for (dist, label) in distances[:k]]
    return Counter(k_nearest).most_common(1)[0][0]

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    # Ajouter les labels
    ax.set_xlabel('Prédictions', fontsize=14)
    ax.set_ylabel('Vraies valeurs', fontsize=14)
    ax.set_title('Matrice de Confusion des Gestes', fontsize=16, pad=20)
    
    ax.xaxis.set_ticklabels([f'Geste {i}' for i in range(10)], rotation=45)
    ax.yaxis.set_ticklabels([f'Geste {i}' for i in range(10)], rotation=0)
    
    plt.tight_layout()
    plt.show()

accs, y_test_all, y_pred_all = Cross_validation(data, model=None, Deep=False, user_dependent=True, k=3)
plot_confusion_matrix(y_test_all, y_pred_all)
print("\n--- Rapport de classification ---")
print(classification_report(y_test_all, y_pred_all, digits=4))


<br>
<font size=6 color=#009999> 5 - Deep learning method (using tensorflow) </font> <br>
<br>

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

model = Sequential([
            Input(shape=(170,)),
            Dense(128, activation='relu'),# 85 * 2
            Dropout(0.3),
            Dense(64, activation='relu'),
            Dense(10, activation='softmax')
        ])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

accs1, y_test_all1, y_pred_all1 = Cross_validation(data, model=model, Deep=True, user_dependent=False,k=3)
accs2, y_test_all2, y_pred_all2 = Cross_validation(data, model=model, Deep=True, user_dependent=True,k=3)

plot_confusion_matrix(y_test_all1, y_pred_all1)
print("\n--- Rapport de classification ---")
print(classification_report(y_test_all1, y_pred_all1, digits=4))

print(classification_report(y_test_all2, y_pred_all2, digits=4))



<br>
<font size=6 color=#009999> 6 - Other datasets (Domain4)</font> <br>
<br>

In [ ]:
domain4_path = "data/Domain4_csv"
files = natsorted(os.listdir(domain4_path))
data = {
    'subjects': [],      # Index du sujet (1-10)
    'gesture_type': [],  # Type de formes (Cone, sphere, ...)
    'repetition': [],    # Numéro de répétition (1-10)
    'trajectory': []     # Tableau numpy des positions [129, 3]
}

scaler = StandardScaler()

form_to_digit = {
    'Tetrahedron': 0,
    'Toroid': 1,
    'CylindricalPipe': 2,
    'Cone': 3,
    'Sphere': 4,
    'Cylinder': 5,
    'RectangularPipe': 6,
    'Pyramid': 7,
    'Hemisphere': 8,
    'Cuboid': 9
}
digit_to_form = {v: k for k, v in form_to_digit.items()} # permet de faire la conversion dans l'autre sens

for filename in files:
    filepath = os.path.join(domain4_path, filename)  
    df = pd.read_csv(filepath, header=None, names=["x", "y", "z", "t"], skiprows=1)

    # Ex: Subject1-Cylinder-2.csv → subject=1, gesture='Cylinder', repeat=2
    parts = filename.replace(".csv", "").replace("Subject", "").split("-")
    subject_id = int(parts[0])
    gesture_id = str(parts[1])
    repetition_id = int(parts[2])

    data['subjects'].append(subject_id)
    data['gesture_type'].append(form_to_digit[gesture_id]) # convertis forme en chiffre (ex: Toroid -> 1)
    data['repetition'].append(repetition_id)
    data['trajectory'].append(scaler.fit_transform(df.iloc[:, :3].values.astype(float)))

dataframe = pd.DataFrame(data)

In [ ]:
# import numpy as np
from sklearn import preprocessing
import numpy as np
# Import the dataset and discard non-quantitative features
#dataset01 = pd.read_csv("GestureRecognitionProject\Datasets_CSV\Domain1_csv\Subject1-0-1.csv") # todo other dataset

# Remove the time step (last column)
#dataset01 = dataset01.drop(["<t>"], axis=1)

# Observe the dataset
dataframe.info()
display(dataframe.describe())

#mean length for resizing for Deep method
lengths = [len(traj) for traj in data['trajectory']]
target_len = int(np.median(lengths))
print(target_len)

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt

#### PCA #####
# Stack all points from all files
concatenated = np.vstack(data['trajectory'])  # shape: (total_points, 3)

# Step 2: Standardize all data (already done)
scaled_points = concatenated

# Step 3: Fit PCA on scaled data
pca = PCA(n_components=2)
pca.fit(scaled_points)

# Step 4: Apply PCA to each sample
#projected_samples = [pca.transform(seq) for seq in data['trajectory']]
data['trajectory'] = [pca.transform(seq) for seq in data['trajectory']]
# comment to choose if we want to go with PCA or only scale
#data['trajectory'] = projected_sample # PCA
#data['trajectory'] = [seq[:, :-1] for seq in data['trajectory']] # only scale (we take x,y)
#### PCA #####

labels = data['gesture_type']

#comment to change what is visualize in the graph
X_pca = np.vstack(scaled_points)

#X_pca = np.vstack(projected_samples) #bizarre mais global PCA rends moins bien visible les chiffres
fig, axs = plt.subplots(2, 5, figsize=(15, 6))
axs = axs.ravel()

for form in range(10):
    indices = [i for i, g in enumerate(data['gesture_type']) if g == form]
    for idx in indices[:9]:  # jusqu’à 5 répétitions par geste
        traj = data['trajectory'][idx]
        axs[form].plot(traj[:, 0], traj[:, 1], alpha=0.6)
    axs[form].set_title(f"Forme {digit_to_form[form]}")
    axs[form].axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Test avec dtw + K-NN
accs, y_test_all, y_pred_all = Cross_validation(data, model=None, Deep=False, user_dependent=True, k=3)

In [ ]:
# Résultats pour dtw + K-NN
y_test_all_labels = [digit_to_form[label] for label in y_test_all]
y_pred_all_labels = [digit_to_form[label] for label in y_pred_all]
labels_to_use = ['Tetrahedron','Toroid','CylindricalPipe','Cone','Sphere','Cylinder', 'RectangularPipe','Pyramid','Hemisphere','Cuboid']

cm = confusion_matrix(y_test_all_labels, y_pred_all_labels, labels=labels_to_use)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_to_use, yticklabels=labels_to_use, cbar=False)
plt.title("Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()
print("\n--- Rapport de classification ---")
print(classification_report(y_test_all, y_pred_all, digits=4))

In [ ]:
# Test with deep model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

model = Sequential([
            Input(shape=(170,)),
            Dense(128, activation='relu'),
            Dropout(0.3),
            Dense(64, activation='relu'),
            Dense(10, activation='softmax')
        ])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

accs1, y_test_all1, y_pred_all1 = Cross_validation(data, model=model, Deep=True, user_dependent=False,k=3)
accs2, y_test_all2, y_pred_all2 = Cross_validation(data, model=model, Deep=True, user_dependent=True,k=3)

In [ ]:
# Résultats pour deep model, user_dependent = False

y_test_all1_labels = [digit_to_form[label] for label in y_test_all1]
y_pred_all1_labels = [digit_to_form[label] for label in y_pred_all1]
labels_to_use = ['Tetrahedron','Toroid','CylindricalPipe','Cone','Sphere','Cylinder', 'RectangularPipe','Pyramid','Hemisphere','Cuboid']

cm = confusion_matrix(y_test_all1_labels, y_pred_all1_labels, labels=labels_to_use)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_to_use, yticklabels=labels_to_use, cbar=False)
plt.title("Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

print("\n--- Classification Report ---")
print(classification_report(y_test_all1_labels, y_pred_all1_labels, target_names=labels_to_use, digits=4))


In [ ]:
# Résultats pour deep model, user_dependent = True

y_test_all2_labels = [digit_to_form[label] for label in y_test_all2]
y_pred_all2_labels = [digit_to_form[label] for label in y_pred_all2]
labels_to_use = ['Tetrahedron','Toroid','CylindricalPipe','Cone','Sphere','Cylinder', 'RectangularPipe','Pyramid','Hemisphere','Cuboid']

cm = confusion_matrix(y_test_all2_labels, y_pred_all2_labels, labels=labels_to_use)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_to_use, yticklabels=labels_to_use, cbar=False)
plt.title("Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

print("\n--- Rapport de classification ---")
print(classification_report(y_test_all2, y_pred_all2, digits=4))